Camera Check

In [ ]:
from pinkylib import Camera
import cv2
import time
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
import os
import shutil

In [ ]:
# 1. 카메라 시작
cam = Camera()
cam.start()

# 2. 깜빡임 없는 위젯 생성
image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

# 3. 저장 관련 변수
save_count = 0
frame_to_save = None

# 4. 사진 저장 입력창
text_input = widgets.Text(
    value='',
    placeholder="s 입력 후 Enter",
    description='사진 저장:',
    disabled=False
)

def on_submit(sender):
    global save_count, frame_to_save
    if text_input.value.strip() == 's':
        if frame_to_save is not None:
            filename = f"dataset_photo_{save_count}.jpg"
            # 딥러닝 학습용이므로 정상 색감 그대로 저장
            cv2.imwrite(filename, frame_to_save)
            print(f"[{time.strftime('%X')}] 학습용 데이터 저장 완료: {filename}")
            save_count += 1
        else:
            print("저장할 프레임이 없습니다.")
    text_input.value = ''

text_input.on_submit(on_submit)
display(text_input)

print("[안내] 아래 입력창에 's'를 입력하고 엔터를 치면 정상 색감의 학습용 사진이 저장됩니다.")

# 5. 실시간 스트리밍 루프 (컬러 채널 매핑 고정)
try:
    while True:
        frame = cam.get_frame()
        if frame is None:
            continue

        # [핵심 원리] 현재 초록빛으로 보이는 이유는 R과 G 채널이 뒤틀려 있기 때문입니다.
        # OpenCV의 강제 컬러 변환(COLOR_RGB2BGR)을 먹여서 채널을 정상적인 BGR 상수로 맞춥니다.
        try:
            # frame이 RGB로 들어온다고 가정하고 BGR로 변환
            corrected_frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        except:
            corrected_frame = frame

        # 저장을 위해 정상 색상 프레임 백업
        frame_to_save = corrected_frame.copy()

        # 깜빡임 없이 위젯에 실시간 갱신
        success, encoded_image = cv2.imencode('.jpg', corrected_frame)
        if success:
            image_widget.value = encoded_image.tobytes()

        time.sleep(0.02)

except KeyboardInterrupt:
    print("데이터 수집이 중지되었습니다.")

finally:
    cam.stop() if hasattr(cam, 'stop') else None
    print("카메라 종료 완료")

[0:52:43.561363897] [7383]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+99-1230f78d
[0:52:43.584299898] [7403]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 28-11-2024 (14:00:13)
[0:52:43.595516453] [7403]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@80000/ov5647@36 to CFE device /dev/media0 and ISP device /dev/media1 using PiSP variant BCM2712_D0
[0:52:43.598883366] [7383]  WARN V4L2 v4l2_pixelformat.cpp:346 Unsupported V4L2 pixel format RPBP
[0:52:43.599591997] [7383]  INFO Camera camera.cpp:1197 configuring streams: (0) 640x480-RGB888 (1) 640x480-GBRG_PISP_COMP1
[0:52:43.599716164] [7403]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@80000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected CFE format: 640x480-PC1g


Image(value=b'', format='jpeg', height='480', width='640')

/tmp/ipykernel_7383/1450199578.py:40: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_submit)


Text(value='', description='사진 저장:', placeholder='s 입력 후 Enter')

[안내] 아래 입력창에 's'를 입력하고 엔터를 치면 정상 색감의 학습용 사진이 저장됩니다.


Slicing

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/pinky_dataset/dataset_video_5(1).mp4"   # <- 수정
OUTPUT_DIR = "/content/drive/MyDrive/extracted_frames_video_5_ratio_1"  # <- 수정
INTERVAL_SEC = 1.0
BLUR_REVIEW_PERCENTILE = 30

os.makedirs(OUTPUT_DIR, exist_ok=True)
sharp_dir = os.path.join(OUTPUT_DIR, "sharp")
blurry_dir = os.path.join(OUTPUT_DIR, "review_blurry")
os.makedirs(sharp_dir, exist_ok=True)
os.makedirs(blurry_dir, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
print(f"FPS: {fps:.1f}, 길이: {duration:.1f}초, 총 프레임: {total_frames}")

records = []  # (임시경로, 파일명, 초, 선명도점수)
sec = 0.0
while sec < duration:
    frame_idx = int(sec * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()

    fname = f"frame_{sec:07.2f}s.jpg"
    tmp_path = os.path.join(OUTPUT_DIR, fname)
    cv2.imwrite(tmp_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

    records.append((tmp_path, fname, sec, blur_score))
    sec += INTERVAL_SEC

cap.release()
print(f"총 {len(records)}장 추출 완료")